# BLAST

This notebook demonstrates the two BLAST tools: `run_create_blast_db` builds a local BLAST database from a FASTA file, and `run_blast_search` searches sequences against either a local database or NCBI's online servers. BLAST+ binaries are downloaded automatically on first use, so no manual installation is required.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("blast")
display_overview("blast")
display_docs_section("blast", "Background")

# BLAST

> BLAST (Basic Local Alignment Search Tool) finds regions of similarity between biological sequences. It compares nucleotide or protein sequences to sequence databases and calculates the statistical significance of matches. This module provides a unified interface for both *Online BLAST* (querying NCBI servers remotely) and *Local BLAST+* (running searches against custom or downloaded databases on your own hardware), as well as utilities for creating custom BLAST databases.

## Background

**What does this tool do?**
[BLAST](https://blast.ncbi.nlm.nih.gov/Blast.cgi) finds regions of local similarity between sequences by comparing nucleotide or protein sequences to sequence databases like [NCBI GenBank](https://www.ncbi.nlm.nih.gov/genbank/) and calculates the statistical significance of matches.

**Why is this important?**
Sequence alignment is the first step in almost all bioinformatics workflows. It allows researchers to:

* Infer functional relationships: If a new sequence resembles a known gene, they likely share a function.
* Identify species: Map unknown DNA reads to specific organisms (metagenomics).
* Detect off-targets: Ensure a primer or CRISPR guide only binds to the intended target.
* Find evolutionary origins: Trace the phylogeny of a gene across species.

**Scientific foundation:**
BLAST uses a heuristic algorithm that seeks high-scoring segment pairs (HSPs). It does not perform a full [Smith-Waterman](https://en.wikipedia.org/wiki/Smith%E2%80%93Waterman_algorithm) alignment (which is accurate but slow). Instead, it does:

1. **Seeding**: Breaks the query into short "words" (k-mers) and finds exact matches in the database.
2. **Extension**: Extends these matches in both directions until the alignment score drops below a threshold.
3. **Evaluation**: Calculates an [E-value](https://en.wikipedia.org/wiki/BLAST_\(biotechnology\)#Algorithm) (Expect value) based on the Karlin-Altschul statistics, representing the number of hits one can expect to see by chance.

## Available tools

In [2]:
display_available_tools("blast")

- **`run_create_blast_db()`** — Create a local BLAST database from a FASTA file
- **`run_blast_search()`** — Search sequences against BLAST databases (online or local)

### `run_create_blast_db`

`run_create_blast_db` builds a local BLAST database from a FASTA file using the `makeblastdb` utility from BLAST+. It produces a set of indexed database files whose base path can then be passed directly to `BlastSearchConfig.local_db`. Both nucleotide and protein databases are supported via the `dbtype` parameter.

In [3]:
import tempfile
from pathlib import Path

from proto_tools import (
    CreateBlastDbInput,
    CreateBlastDbConfig,
    run_create_blast_db,
)

In [4]:
# Display input docs
display_api_reference("blast", "input", "run_create_blast_db")

# Write a small FASTA with three well-characterized human globin proteins
tmp_dir = tempfile.mkdtemp(prefix="blast_example_")

fasta_path = Path(tmp_dir) / "test_sequences.fasta"
fasta_path.write_text(
    ">sp|P69905|HBA_HUMAN Hemoglobin subunit alpha\n"
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH\n"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKL\n"
    "LSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR\n"
    ">sp|P68871|HBB_HUMAN Hemoglobin subunit beta\n"
    "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST\n"
    "PDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDP\n"
    "ENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH\n"
    ">sp|P02144|MYG_HUMAN Myoglobin\n"
    "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHL\n"
    "KSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIP\n"
    "VKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG\n"
)
print(f"Wrote test FASTA to: {fasta_path}")

db_input = CreateBlastDbInput(fasta=str(fasta_path))

**Input** — `CreateBlastDbInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `fasta` | `string` | required | Path to a FASTA file containing the sequences to be indexed into a BLAST database. The file must exist and contain valid FASTA-formatted sequences. For nucleotide databases, sequences should be DNA or RNA. For protein databases, sequences should be amino acids. |

Wrote test FASTA to: /tmp/blast_example_07zbd3cm/test_sequences.fasta


In [5]:
# Display config docs
display_api_reference("blast", "config", "run_create_blast_db")

# Create a protein database with a descriptive title | see docs above for all fields
db_config = CreateBlastDbConfig(dbtype="prot", title="Test Protein DB")

**Config** — `CreateBlastDbConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `dbtype` | `enum` | `nucl` | The type of database to create: Available options: `nucl`, `prot` |
| `out_prefix` | `string` |  | Optional file path prefix for the generated database files. If not specified, the database files will be created in the same directory as the input FASTA file, using the FASTA filename (without extension) as the prefix. For example, if the input is `"sequences.fasta"` and `out_prefix` is `None`, the database will be named `"sequences"`. If `out_prefix` is `"/data/mydb"`, the database files will be created as `"/data/mydb.nhr"`, `"/data/mydb.nin"`, `"/data/mydb.nsq"`, etc. |
| `title` | `string` |  | Optional descriptive title for the database. This title will be displayed in BLAST search results and can help identify the database contents. If not specified, `makeblastdb` will use the input filename. |
| `additional_params` | `Dict[string, string | integer | number | boolean]` |  | Dictionary of additional parameters for `makeblastdb`. Common options include: |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run database creation
db_result = run_create_blast_db(db_input, db_config)

Running run_create_blast_db [00:00]

In [7]:
# Display output docs
display_api_reference("blast", "output", "run_create_blast_db")

# The db_path is passed directly to BlastSearchConfig.local_db for searches
print(f"Database created at: {db_result.db_path}")

**Output** — `CreateBlastDbOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `db_path` | `string` | required | The base path to the generated BLAST database files (without file extensions). This path can be used directly as the value for the `local_db` parameter in `BlastSearchConfig`. For example, if `db_path` is `"/data/mydb"`, `makeblastdb` will have created multiple files like `"/data/mydb.nhr"`, `"/data/mydb.nin"`, `"/data/mydb.nsq"` (for nucleotide databases) or similar extensions for protein databases. |

Database created at: /tmp/blast_example_07zbd3cm/test_sequences


### `run_blast_search`

`run_blast_search` performs sequence similarity search in either local or online mode. Local mode runs BLAST+ against a database created by `run_create_blast_db`, while online mode submits the query to NCBI's QBLAST service and returns hits from large public databases such as SwissProt or nr. The query accepts either a raw sequence string or a path to a FASTA file, and results are returned as a list of `BlastHit` objects with the standard 12-column tabular fields.

In [8]:
from proto_tools import (
    BlastSearchInput,
    BlastSearchConfig,
    run_blast_search,
)

In [9]:
# Display input docs
display_api_reference("blast", "input", "run_blast_search")

# Write a FASTA query file with the hemoglobin beta sequence
query_seq = "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH"

query_path = Path(tmp_dir) / "query.fasta"
query_path.write_text(f">query_hbb\n{query_seq}\n")

search_input = BlastSearchInput(query=str(query_path))

**Input** — `BlastSearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `query` | `string` | required | A raw nucleotide/protein sequence (e.g. `"ATGCGTAAA"`) or a path to a FASTA file. |
| `query_type` | `enum` | `sequence` | Automatically set to `"sequence"` or `"fasta_path"` during validation. Read-only; do not set manually. Available options: `sequence`, `fasta_path` |

In [10]:
# Display config docs
display_api_reference("blast", "config", "run_blast_search")

# Local blastp search against the database we just built | see docs above for all fields
search_config = BlastSearchConfig(
    search_mode="local",
    program="blastp",
    local_db=db_result.db_path,
    num_threads=2,
)

**Config** — `BlastSearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `search_mode` | `enum` | `online` | `"online"` routes to NCBI QBLAST; `"local"` runs BLAST+ CLI against a local database. Available options: `online`, `local` |
| `program` | `enum` | `blastn` | BLAST algorithm (blastn, blastp, blastx, tblastn, tblastx). Available options: `blastn`, `blastp`, `blastx`, `tblastn`, `tblastx` |
| `database` | `enum` | `nt` | NCBI database to search (online mode only). Available options: `nt`, `nr`, `refseq_rna`, `refseq_protein`, `swissprot`, `pdb`, `pataa`, `patnt` |
| `entrez_query` | `string` |  | Restrict online search with an Entrez query. |
| `hitlist_size` | `integer` |  | Number of hits to return (online mode only). |
| `megablast` | `boolean` |  | Use MegaBLAST algorithm (online mode, blastn only). |
| `local_db` | `string` |  | Path to a local BLAST database (local mode only, required). |
| `num_threads` | `integer` | `4` | CPU threads for local search. |
| `evalue` | `number` |  | E-value threshold. |
| `word_size` | `integer` |  | Word size for initial matches. |
| `gapopen` | `integer` |  | Cost to open a gap. |
| `gapextend` | `integer` |  | Cost to extend a gap. |
| `matrix` | `string` |  | Scoring matrix for protein searches. |
| `reward` | `integer` |  | Nucleotide match reward (blastn only). |
| `penalty` | `integer` |  | Nucleotide mismatch penalty (blastn only). |
| `threshold` | `integer` |  | Min word score for lookup table (protein only). |
| `comp_based_stats` | `integer` |  | Composition-based statistics mode (protein only). |
| `max_target_seqs` | `integer` |  | Max aligned sequences to keep. |
| `max_hsps` | `integer` |  | Max HSPs per query-subject pair. |
| `perc_identity` | `number` |  | Minimum percent identity filter. |
| `qcov_hsp_perc` | `number` |  | Minimum query coverage per HSP. |
| `culling_limit` | `integer` |  | Delete hits enveloped by better hits. |
| `best_hit_overhang` | `number` |  | Best-hit algorithm overhang value. |
| `best_hit_score_edge` | `number` |  | Best-hit algorithm score edge value. |
| `subject_besthit` | `boolean` |  | Only report best hit per subject. |
| `soft_masking` | `boolean` |  | Use soft masking for initial matches. |
| `lcase_masking` | `boolean` |  | Treat lowercase in FASTA as masked. |
| `dust` | `string` |  | Low-complexity filter for nucleotide queries (blastn only). |
| `seg` | `string` |  | Low-complexity filter for protein queries. |
| `task` | `string` |  | BLAST task for optimized defaults. |
| `ungapped` | `boolean` |  | Perform ungapped alignment only. |
| `strand` | `string` |  | Query strand(s) to search (nucleotide queries only). |
| `query_gencode` | `integer` |  | Genetic code for translating query (blastx/tblastx). |
| `db_gencode` | `integer` |  | Genetic code for translating DB (tblastn/tblastx). |
| `window_size` | `integer` |  | Multiple-hits window size. |
| `xdrop_ungap` | `number` |  | X-dropoff for ungapped extensions. |
| `xdrop_gap` | `number` |  | X-dropoff for preliminary gapped extensions. |
| `xdrop_gap_final` | `number` |  | X-dropoff for final gapped alignment. |
| `use_sw_tback` | `boolean` |  | Compute Smith-Waterman alignments (protein only). |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [11]:
# Run the local BLAST search
result = run_blast_search(search_input, search_config)

Running run_blast_search [00:00]

In [12]:
# Display output docs
display_api_reference("blast", "output", "run_blast_search")

# HBB should self-hit at 100% identity; HBA should appear at ~43% identity
print(f"Found {result.num_hits} hits\n")
for hit in result.hits:
    print(f"  {hit.sseqid}: {hit.pident:.1f}% identity, e-value={hit.evalue:.1e}, bitscore={hit.bitscore:.1f}")

**Output** — `BlastSearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `hits` | `List[BlastHit]` |  | BLAST alignment hits with standard tabular |
| `qseqid` | `string` | required | Query sequence ID. |
| `sseqid` | `string` | required | Subject sequence ID. |
| `pident` | `number` | required | Percentage of identical matches. |
| `length` | `integer` | required | Alignment length. |
| `mismatch` | `integer` | required | Number of mismatches. |
| `gapopen` | `integer` | required | Number of gap openings. |
| `qstart` | `integer` | required | Start of alignment in query. |
| `qend` | `integer` | required | End of alignment in query. |
| `sstart` | `integer` | required | Start of alignment in subject. |
| `send` | `integer` | required | End of alignment in subject. |
| `evalue` | `number` | required | Expect value. |
| `bitscore` | `number` | required | Bit score. |

Found 2 hits

  sp|P68871|HBB_HUMAN: 100.0% identity, e-value=1.4e-111, bitscore=301.0
  sp|P69905|HBA_HUMAN: 43.4% identity, e-value=5.4e-38, bitscore=114.0


#### Online search

For searches against large public databases, set `search_mode="online"`. The query is submitted to NCBI's QBLAST service and requires an internet connection. The raw sequence string can be passed directly to `BlastSearchInput` without writing a FASTA file.

In [13]:
# Online search accepts a raw sequence string directly
online_input = BlastSearchInput(query=query_seq)
online_config = BlastSearchConfig(
    search_mode="online",
    program="blastp",
    database="swissprot",
    hitlist_size=5,
)

online_result = run_blast_search(online_input, online_config)
print(f"Found {online_result.num_hits} hits\n")
for hit in online_result.hits:
    print(f"  {hit.sseqid}: {hit.pident:.1f}% identity, e-value={hit.evalue:.1e}, bitscore={hit.bitscore:.1f}")

Found 5 hits

  P68871: 100.0% identity, e-value=5.8e-106, bitscore=301.2
  P02024: 99.3% identity, e-value=2.0e-105, bitscore=300.1
  P02025: 98.6% identity, e-value=3.0e-103, bitscore=294.3
  P02032: 97.3% identity, e-value=2.4e-102, bitscore=292.0
  P19885: 95.9% identity, e-value=4.3e-102, bitscore=291.6


## Export Results

Search results can be exported to CSV or JSON for downstream analysis. CSV produces one row per hit with the standard 12 tabular fields; JSON preserves the same structure for programmatic consumption.

In [14]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="blast_search_local_results", export_path=output_dir, file_format="csv")
online_result.export(name="blast_search_online_results", export_path=output_dir, file_format="csv")
print(f"Results exported to {output_dir}")

Results exported to example_output
